In [1]:
# =============================================================================
# data.ipynb — Loading & Normalization
# =============================================================================
# Purpose
# -------
# Prepare the short-format (resumen) and long-format (texto) corpora for
# retrieval experiments. We:
#   1) Load objects from two json (short/long).
#   2) Normalize text (lowercase, whitespace), preserving numerals.
#   3) Build two views per item:
#        - text_word : tokenized string for word-based features
#        - text_char : clean, padded string for character n-grams (char_wb 3–5)
#   4) Extract a `numbers` list from each text (for numeric-match features).
#   5) Show a capped PREVIEW joining short↔long on `item_key` (not `id`).
#   6) Optionally save processed outputs (off by default).
#
# Inputs (expected)
# -----------------
# - Two json files with lists of LlamaIndex-like `Document` objects:
#     /work/data/processed/{COLLECTION}_resumen.json
#     /work/data/processed/{COLLECTION}_texto.json
#   Each Document should have:
#     - doc.text        : the body text (string)
#     - doc.metadata    : dict with at least:
#         item_key, parent_key, ud, concept, parameters
#   NOTE: `id` can differ between the two lists. We use `item_key` as the
#   reliable join key throughout this notebook.
#
# Outputs (derived columns added to DataFrames)
# ---------------------------------------------
# - text_norm   : normalized string (lowercased, whitespace-collapsed,
#                 numerals standardized to '.' for decimals, '%' spaced)
# - tokens_word : list[str] word-level tokens (Spanish letters + numeric forms)
# - text_word   : "tokens_word" joined by spaces (for TF–IDF/BM25 word features)
# - text_char   : clean padded string: " {text_norm} " (for char_wb n-grams)
# - numbers     : list[float] of numerals extracted from text_norm
#
# Normalization details (Spanish-oriented)
# ----------------------------------------
# - Case folding: lower()
# - Whitespace: collapse runs to single spaces and strip ends
# - Numerals:
#     * "1,60" → "1.60" (decimal comma → dot)
#     * "1.234" → "1234" (remove thousands dot when pattern = \d.\d{3}\b)
#     * "95%" → "95 %" (separate percent sign)
#
# Tokenization & Fields
# ---------------------
# - Word tokens (regex):
#     [a-záéíóúüñ]+ | \d+(\.\d+)?(x\d+(\.\d+)?)?
#   This keeps Spanish letters and numeric forms like "4x40".
# - text_word:
#     tokens joined with spaces → ideal for word-level TF–IDF/BM25.
# - text_char:
#     a plain *string* derived from text_norm with boundary padding:
#       f" {text_norm} "
#     Use later with scikit-learn:
#       TfidfVectorizer(analyzer="char_wb", ngram_range=(3,5))
#   NOTE: We intentionally avoid per-token markers like ^token$ here.
#
# Numbers extraction (for later numeric-match features)
# -----------------------------------------------------
# - Captures integers/decimals and multiplicatives:
#     "4x40" → [4.0, 40.0]
# - After normalization, decimals use '.' (e.g., 1.60).
# - We do not store units; only numeric values (e.g., "95 %" → 95.0).
#
# Preview behavior
# ----------------
# - We preview a small, deterministic head of the joined short↔long table.
# - Join key: `item_key` (string-stripped on both sides).
# - Optional per-side de-duplication by `item_key` can be enabled if needed.
# - PREVIEW_SAMPLE controls the number of rows shown (default 25).
#
# Saving (preview-first)
# ----------------------
# - By default, SAVE = False (no files written until you inspect the preview).
# - When SAVE = True, we write to OUT_DIR in OUT_FORMAT:
#     * parquet (preferred; requires pyarrow/fastparquet)
#     * csv
#     * jsonl
# - Filenames:
#     {COLLECTION}_short_norm.{parquet|csv|jsonl}
#     {COLLECTION}_long_norm.{parquet|csv|jsonl}
#
# Config knobs (set near the top of the notebook)
# -----------------------------------------------
#   COLLECTION     : e.g., "OEB"
#   DATA_DIR       : e.g., Path("/work/data/processed")
#   PREVIEW_SAMPLE : e.g., 25
#   SAVE           : False by default; set True to write outputs
#   OUT_DIR        : where to save (default = DATA_DIR)
#   OUT_FORMAT     : "parquet" | "csv" | "jsonl"
#
# Assumptions / Non-goals
# -----------------------
# - We do NOT auto-map arbitrary column names → upstream must provide:
#     item_key, parent_key, ud, concept, parameters, text
# - We do NOT rely on `id` for alignment; different Document lists may
#   carry different ids. `item_key` is the canonical join key.
# - Notebook is optimized for ~40k items per split; previews are capped to
#   avoid flooding the UI.
#
# Troubleshooting
# ---------------
# - ModuleNotFoundError: 'caas_jupyter_tools'
#     → We use plain IPython display; no special tools required.
# - AttributeError: 'list' object has no attribute 'columns'
#     → Your pickle likely contains a list of LlamaIndex Documents.
#       We convert those lists into a DataFrame before processing.
# - 0 rows after merge
#     → Ensure we join on `item_key`, and strip whitespace on both sides.
# - Parquet write failure
#     → Install pyarrow or fastparquet, or switch OUT_FORMAT to "csv"/"jsonl".
#
# Repro tip
# ---------
# - Keep this notebook "preview-first". Only enable SAVE after you confirm
#   the preview looks correct on your corpus snapshot.
# =============================================================================


In [1]:
# data.ipynb — Loading & normalization (preview-first, no save by default)
#
# Assumptions per user:
# - User provides the collection name (e.g., "OEB").
# - Exact filenames exist: /data/processed/{COLLECTION}_resumen.pkl and /data/processed/{COLLECTION}_texto.pkl
# - No need to handle .pkl typos or infer collection, and no schema Auto-mapping.
#
# What this cell does:
# 1) Load short & long pickles.
# 2) Normalize text (lowercase, whitespace), standardize numerals (keep digits).
# 3) Produce text_word, text_char, numbers list.
# 4) Show a PREVIEW (limited sample) to avoid notebook overflow.
# 5) Provide a 'save_outputs()' function, but DO NOT SAVE until you set SAVE=True and re-run.
#
# Columns expected in both pickles (at minimum):
# ['id', 'item_key', 'parent_key', 'ud', 'concept', 'parameters', 'text']
#
# Tip: If your columns differ, please align them upstream; this notebook will not remap names.

import os
import re
import json
from pathlib import Path
from typing import Iterable, List, Dict, Any, Union, Optional, Tuple
import pandas as pd
import math
import copy

# ---------------------- CONFIG ----------------------
COLLECTION = "OEB"        # <- user-provided
#project_root = Path('work')
#DATA_DIR = project_root / "data" / "processed"
DATA_DIR =  Path("/work/data/processed")
SHORT_FILE = DATA_DIR / f"{COLLECTION}_resumen.json"
LONG_FILE  = DATA_DIR / f"{COLLECTION}_texto.json"

# Preview & saving controls
PREVIEW_SAMPLE = 25       # limit rows in preview to avoid overflow
SAVE = True               # set to True to save after preview
OUT_DIR = DATA_DIR        # default output directory when saving
OUT_FORMAT = "parquet"    # "parquet" | "csv" | "jsonl"

# --- LOADER FOR LLAMA-INDEX-like Document LISTS -> DataFrame ---
def _as_record_from_obj(doc: Any) -> Dict[str, Any]:
    """
    Accepts dicts like:
      {"text": "...", "metadata": {...}}  OR  {"text": "...", "item_key": "...", ...}
    Falls back to top-level keys if metadata is missing.
    """
    if isinstance(doc, dict):
        text = doc.get("text") or doc.get("content")
        md   = (doc.get("metadata") or {}) if isinstance(doc.get("metadata"), dict) else {}
        root = doc
        doc_id = root.get("id") or root.get("id_") or root.get("doc_id") \
                 or md.get("id") or md.get("id_") or md.get("doc_id")
    else:
        # attribute-style (kept for compatibility)
        text = getattr(doc, "text", None)
        md   = getattr(doc, "metadata", None) or {}
        root = {}
        doc_id = getattr(doc, "id_", None) or getattr(doc, "doc_id", None) or md.get("id")

    def pick(key: str, default=None):
        # prefer metadata, then top-level
        return md.get(key, root.get(key, default))

    # Ensure parameters is a dict for downstream normalization
    params = pick("parameters", {})
    if not isinstance(params, dict):
        params = {}

    return {
        "id": doc_id,
        "item_key": pick("item_key"),
        "parent_key": pick("parent_key"),
        "ud": pick("ud"),
        "concept": pick("concept"),
        "parameters": params,
        "text": text if isinstance(text, str) else pick("text", ""),
    }

def load_llamaindex_docs(path: Path) -> pd.DataFrame:
    """  Load from:
      - .json: a list of documents OR an object with a top-level 'documents' list
    Each element should be a LlamaIndex-like Document (dict) or already-flat dict with 'text' & 'metadata'."""
    
    with open(path, "r", encoding="utf-8") as f:
        obj = json.load(f)
        if isinstance(obj, list):
            rows = [_as_record_from_obj(d) for d in obj]
        elif isinstance(obj, dict):
            # allow {"documents": [...]} or {"data":[...]} shapes
            container = None
            for key in ("documents", "data", "items", "records"):
                if key in obj and isinstance(obj[key], list):
                    container = obj[key]
                    break
            if container is None:
                raise TypeError(f"Unsupported JSON shape in {path}. Expected list or an object with a list under 'documents'/'data'.")
            rows = [_as_record_from_obj(d) for d in container]
        else:
            raise TypeError(f"Unsupported JSON root type in {path}: {type(obj)}")
  
    df = pd.DataFrame(rows)  
    return df.reset_index(drop=True)

# ---------------- Normalization & tokenization ----------------
WS_RE = re.compile(r"\s+", re.UNICODE)
THOUSANDS_DOT_RE = re.compile(r"(?<=\d)\.(?=\d{3}\b)")  # 1.234 -> 1234
DECIMAL_COMMA_RE = re.compile(r"(?<=\d),(?=\d)")        # 1,60 -> 1.60
PERCENT_RE = re.compile(r"(?<=\d)%")                    # 95%  -> 95 %
WORD_TOKEN_RE_NO_NUM = re.compile(r"[a-záéíóúüñ]+" , flags=re.UNICODE) # tokens used to decide "multi-word"
_WORD = re.compile(r"[a-záéíóúüñ0-9]+", re.UNICODE)

def clean_df(df: pd.DataFrame) -> pd.DataFrame:
    # Drop rows with empty/NaN text
    df = df.dropna(subset=["text", "item_key"])
    df = df[df["text"].str.strip() != ""]
    df = df[~df["item_key"].astype(str).str.endswith(("#", "$"), na=False)]
    return df

def normalize_param_string(s: str) -> str:
    """Same normalization as text, then trim inner spaces."""
    return normalize_text(s)

def is_multiword_norm(s: str, min_tokens: int = 2) -> bool:
    """Count *word* tokens (letters only). 'Volumen escaso' -> 2 (True)."""
    return len(WORD_TOKEN_RE_NO_NUM.findall(s)) >= min_tokens

def snake_merge(s: str) -> str:
    """Turn normalized phrase into snake_case, keeping accents."""
    return "_".join(WORD_TOKEN_RE.findall(s))

def normalize_text(s: str) -> str:
    if not isinstance(s, str):
        s = "" if s is None else str(s)
    s = s.lower()
    s = THOUSANDS_DOT_RE.sub("", s)
    s = DECIMAL_COMMA_RE.sub(".", s)
    s = PERCENT_RE.sub(" %", s)
    s = WS_RE.sub(" ", s).strip()
    return s

# Word tokens: Spanish letters & numeric forms, including 4x40
WORD_TOKEN_RE = re.compile(r"[a-záéíóúüñ]+|\d+(?:\.\d+)?(?:x\d+(?:\.\d+)?)?", flags=re.UNICODE)

def tokenize_words(s: str) -> list[str]:
    return WORD_TOKEN_RE.findall(s)

def make_text_char(text_norm: str, pad_char: str = " ") -> str:
    """
    Build the char-ngrams view as a plain normalized string with boundary padding.
    This is what you'll feed to TfidfVectorizer(analyzer='char_wb', ngram_range=(3,5)).
    """
    if not isinstance(text_norm, str):
        text_norm = "" if text_norm is None else str(text_norm)
    return f"{pad_char}{text_norm}{pad_char}"

NUMBER_TOKEN_RE = re.compile(r"\d+(?:\.\d+)?(?:x\d+(?:\.\d+)?)?")

def extract_numbers(s: str) -> list[float]:
    nums = []
    for m in NUMBER_TOKEN_RE.findall(s):
        if "x" in m:
            for p in m.split("x"):
                try:
                    nums.append(float(p))
                except ValueError:
                    pass
        else:
            try:
                nums.append(float(m))
            except ValueError:
                pass
    return nums

def normalize_parameters_field(par):
    """
    Input: nested dict like:
      {'A': {'label': ' TERRENO ', 'values': [{'label': 'a','value': ' blando '}]}, ...}

    Returns:
      parameters_norm:  same structure, but with 'label_norm' and 'value_norm' added
      param_values_multi_norm: flat list[str] of DISTINCT normalized multi-word values (e.g., 'volumen relevante')
    """
    if not isinstance(par, dict):
        return {}, []

    par_norm = copy.deepcopy(par)
    seen = set()
    multi = []

    for _, block in par_norm.items():
        # normalize block label
        raw_label = block.get("label", "")
        block["label_norm"] = normalize_param_string(raw_label)

        # normalize each value entry
        vals = block.get("values", [])
        for v in vals:
            raw_v = v.get("value", "")
            v["value_norm"] = normalize_param_string(raw_v)

            # collect multi-word normalized values (distinct)
            val_norm = v["value_norm"]
            if is_multiword_norm(val_norm):
                if val_norm not in seen:
                    seen.add(val_norm)
                    multi.append(val_norm)

    return par_norm, multi

# === Build tokens from structured parameters (labels + values) ===
# Example output tokens:
#   param_n_tubos_1
#   param_diametros_3
#   param_trabajo_diurno
#   param_banda_de_mantenimiento_i_ge_5_horas
#
# These come from parameters_norm (already produced in add_processed_columns),
# so we don't hardcode any units — we just reuse your normalized label/value text.

def _slug_words(s: str) -> str:
    """Lowercase and keep only [a-záéíóúüñ0-9]+ chunks, joined by '_'."""
    if not isinstance(s, str):
        s = "" if s is None else str(s)
    return "_".join(_WORD.findall(s.lower()))

def _canon_comparators(s: str) -> str:
    """Turn symbols into words so they survive word tokenization."""
    s = s.replace("≥", ">=").replace("≤", "<=")
    s = re.sub(r"\s*>=\s*", " ge ", s)
    s = re.sub(r"\s*<=\s*", " le ", s)
    s = re.sub(r"\s*>\s*",  " gt ", s)
    s = re.sub(r"\s*<\s*",  " lt ", s)
    return s

def build_param_tokens(parameters_norm: dict) -> list[str]:
    """
    For each block in parameters_norm, mint 'param_<label_slug>_<value_slug>'.
    Uses your normalized labels/values, so it's exhaustive (no unit hardcoding).
    """
    out = []
    if not isinstance(parameters_norm, dict):
        return out
    for _, block in parameters_norm.items():
        lab_slug = _slug_words(block.get("label_norm", "")) or "label"
        for v in block.get("values", []):
            val_norm = _canon_comparators(v.get("value_norm", ""))
            val_slug = _slug_words(val_norm) or "value"
            out.append(f"param_{lab_slug}_{val_slug}")
    return out


def add_processed_columns(df: pd.DataFrame, raw_col: str) -> pd.DataFrame:
    df = df.copy()
    df["text_norm"]  = df[raw_col].map(normalize_text)
    df["tokens_word"] = df["text_norm"].map(tokenize_words)
    df["text_word"]   = df["tokens_word"].map(lambda toks: " ".join(toks))
    # OLD (remove): df["text_char"] = df["tokens_word"].map(tokens_to_char_padded)
    # NEW (use the normalized string with boundary padding):
    df["text_char"]   = df["text_norm"].map(make_text_char)
    df["numbers"]     = df["text_norm"].map(extract_numbers)
    out_par_norm = []
    out_param_values_multi = []
    for par in df["parameters"].tolist():
        par_norm, multi_norm = normalize_parameters_field(par)
        out_par_norm.append(par_norm)
        out_param_values_multi.append(multi_norm)

    df["parameters_norm"] = out_par_norm                   # same nested shape + *_norm keys
    df["param_values_multi_norm"] = out_param_values_multi # flat list[str], e.g. ['volumen relevante','sin reposición']
    
    return df

# ---------------- Load, process, PREVIEW ----------------
short_df = load_llamaindex_docs(SHORT_FILE)
long_df  = load_llamaindex_docs(LONG_FILE)

short_df = clean_df(short_df)
long_df  = clean_df(long_df)

# Required columns (no auto-remap as requested)
required = ["item_key", "parent_key", "ud", "concept", "parameters", "text"]
missing_short = [c for c in required if c not in short_df.columns]
missing_long  = [c for c in required if c not in long_df.columns]
if missing_short or missing_long:
    raise KeyError(f"Required columns missing. short: {missing_short} | long: {missing_long}")

# --- PROCESS ---
short_proc = add_processed_columns(short_df, "text")
long_proc  = add_processed_columns(long_df,  "text")

# === Parameter-derived tokens and fused text field ===
short_proc["param_tokens"] = short_proc["parameters_norm"].map(build_param_tokens)
long_proc["param_tokens"]  = long_proc["parameters_norm"].map(build_param_tokens)

# New text field that BM25/TF-IDF can index (keeps original text_word intact)
short_proc["text_word_params"] = short_proc.apply(
    lambda r: (r["text_word"] + " " + " ".join(r["param_tokens"])).strip(), axis=1)
long_proc["text_word_params"] = long_proc.apply(
    lambda r: (r["text_word"] + " " + " ".join(r["param_tokens"])).strip(), axis=1)

# Compact stats
def quick_stats(df: pd.DataFrame, name: str):
    lens = df["tokens_word"].map(len)
    num_counts = df["numbers"].map(len)
    print(f"{name}: rows={len(df)} | tokens avg={lens.mean():.2f} | %with numbers={100*(num_counts>0).mean():.2f}")

quick_stats(short_proc, "SHORT")
quick_stats(long_proc,  "LONG")

# --- PREVIEW (limit to avoid overflow) ---
from IPython.display import display, HTML

def show_preview(df: pd.DataFrame, title: str, rows: int = 25, max_cols: int = 12):
    # cap rows/cols to avoid overflowing the notebook
    n = min(rows, len(df))
    cols = df.columns[:max_cols]
    df_small = df.loc[: n-1, cols].copy()

    # shorten long list columns like 'numbers_*'
    for c in df_small.columns:
        if df_small[c].dtype == object and c.startswith("numbers"):
            df_small[c] = df_small[c].map(lambda x: x if not isinstance(x, list) else (x[:8] + ["…"] if len(x) > 8 else x))

    html = (
        f"<h3 style='margin:8px 0'>{title}</h3>"
        + df_small.to_html(escape=False, index=False, max_rows=n)
    )
    display(HTML(html))

# existing merge result from your code
merged = (
    short_proc[["id","item_key","parent_key","ud","concept","parameters",
                "parameters_norm","param_values_multi_norm",
                "text_word","text_word_params","text_char","numbers","param_tokens"]]
    .merge(
        long_proc[["id","item_key","parameters_norm","param_values_multi_norm",
                   "text_word","text_word_params","text_char","numbers","param_tokens"]],
        on="item_key",
        suffixes=("_short","_long")
    )
)

show_preview(merged, f"Preview ({min(PREVIEW_SAMPLE, len(merged))} rows) — {COLLECTION} short vs long")
print("\nPreview generated. To save outputs, set SAVE=True and re-run this cell.")


# --- OPTIONAL SAVE (off by default) ---
def save_outputs(short_proc: pd.DataFrame, long_proc: pd.DataFrame, out_dir: Path, fmt: str = "parquet"):
    out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    base = COLLECTION
    if fmt == "parquet":
        try:
            short_proc.to_parquet(out_dir / f"{base}_short_norm.parquet", index=False)
            long_proc.to_parquet(out_dir / f"{base}_long_norm.parquet", index=False)
            print("Saved Parquet to:", out_dir)
            return
        except Exception as e:
            print("Parquet failed, falling back to CSV:", e)
            fmt = "csv"
    if fmt == "csv":
        short_proc.to_csv(out_dir / f"{base}_short_norm.csv", index=False)
        long_proc.to_csv(out_dir / f"{base}_long_norm.csv", index=False)
        print("Saved CSV to:", out_dir)
    elif fmt == "jsonl":
        import json
        def to_jsonl(df: pd.DataFrame, path: Path):
            with open(path, "w", encoding="utf-8") as f:
                for _, row in df.iterrows():
                    f.write(json.dumps(row.to_dict(), ensure_ascii=False) + "\n")
        to_jsonl(short_proc, out_dir / f"{base}_short_norm.jsonl")
        to_jsonl(long_proc,  out_dir / f"{base}_long_norm.jsonl")
        print("Saved JSONL to:", out_dir)

if SAVE:
    save_outputs(short_proc, long_proc, OUT_DIR, OUT_FORMAT)


SHORT: rows=47513 | tokens avg=20.78 | %with numbers=99.83
LONG: rows=47513 | tokens avg=84.42 | %with numbers=99.83


id_short,item_key,parent_key,ud,concept,parameters,parameters_norm_short,param_values_multi_norm_short,text_word_short,text_word_params_short,text_char_short,numbers_short
9b41f3b0-9fa6-4640-b124-b4d78e5fabf7,OEB010aaa,OEB010$,m,CANALIZACIÓN PARA LÍNEA SUBTERRANEA DOBLE CIRCUITO DE 220 Ó 400 KV,"{'A': {'label': ' TERRENO ', 'values': [{'label': 'a', 'value': ' blando '}]}, 'B': {'label': ' PAVIMENTO ', 'values': [{'label': 'a', 'value': ' sin reposición '}]}, 'C': {'label': ' CONDICIONES DE EJECUCIÓN', 'values': [{'label': 'a', 'value': 'Volumen relevante'}]}}","{'A': {'label': ' TERRENO ', 'values': [{'label': 'a', 'value': ' blando ', 'value_norm': 'blando'}], 'label_norm': 'terreno'}, 'B': {'label': ' PAVIMENTO ', 'values': [{'label': 'a', 'value': ' sin reposición ', 'value_norm': 'sin reposición'}], 'label_norm': 'pavimento'}, 'C': {'label': ' CONDICIONES DE EJECUCIÓN', 'values': [{'label': 'a', 'value': 'Volumen relevante', 'value_norm': 'volumen relevante'}], 'label_norm': 'condiciones de ejecución'}}","[sin reposición, volumen relevante]",ejecución de canalización para línea subterranea doble circuito de 220 ó 400 kv en terreno blando sin reposición de pavimento volumen relevante,ejecución de canalización para línea subterranea doble circuito de 220 ó 400 kv en terreno blando sin reposición de pavimento volumen relevante param_terreno_blando param_pavimento_sin_reposición param_condiciones_de_ejecución_volumen_relevante,"ejecución de canalización para línea subterranea doble circuito de 220 ó 400 kv en terreno blando , sin reposición de pavimento. (-/-/volumen relevante)","[220.0, 400.0]"
d7990888-6edf-434f-8637-08275194b54a,OEB010aab,OEB010$,m,CANALIZACIÓN PARA LÍNEA SUBTERRANEA DOBLE CIRCUITO DE 220 Ó 400 KV,"{'A': {'label': ' TERRENO ', 'values': [{'label': 'a', 'value': ' blando '}]}, 'B': {'label': ' PAVIMENTO ', 'values': [{'label': 'a', 'value': ' sin reposición '}]}, 'C': {'label': ' CONDICIONES DE EJECUCIÓN', 'values': [{'label': 'b', 'value': 'Volumen escaso'}]}}","{'A': {'label': ' TERRENO ', 'values': [{'label': 'a', 'value': ' blando ', 'value_norm': 'blando'}], 'label_norm': 'terreno'}, 'B': {'label': ' PAVIMENTO ', 'values': [{'label': 'a', 'value': ' sin reposición ', 'value_norm': 'sin reposición'}], 'label_norm': 'pavimento'}, 'C': {'label': ' CONDICIONES DE EJECUCIÓN', 'values': [{'label': 'b', 'value': 'Volumen escaso', 'value_norm': 'volumen escaso'}], 'label_norm': 'condiciones de ejecución'}}","[sin reposición, volumen escaso]",ejecución de canalización para línea subterranea doble circuito de 220 ó 400 kv en terreno blando sin reposición de pavimento volumen escaso,ejecución de canalización para línea subterranea doble circuito de 220 ó 400 kv en terreno blando sin reposición de pavimento volumen escaso param_terreno_blando param_pavimento_sin_reposición param_condiciones_de_ejecución_volumen_escaso,"ejecución de canalización para línea subterranea doble circuito de 220 ó 400 kv en terreno blando , sin reposición de pavimento. (-/-/volumen escaso)","[220.0, 400.0]"
ce0094b4-aab6-402c-9536-f96cf930d7e5,OEB010aac,OEB010$,m,CANALIZACIÓN PARA LÍNEA SUBTERRANEA DOBLE CIRCUITO DE 220 Ó 400 KV,"{'A': {'label': ' TERRENO ', 'values': [{'label': 'a', 'value': ' blando '}]}, 'B': {'label': ' PAVIMENTO ', 'values': [{'label': 'a', 'value': ' sin reposición '}]}, 'C': {'label': ' CONDICIONES DE EJECUCIÓN', 'values': [{'label': 'c', 'value': 'Cualquier condición de ejecución'}]}}","{'A': {'label': ' TERRENO ', 'values': [{'label': 'a', 'value': ' blando ', 'value_norm': 'blando'}], 'label_norm': 'terreno'}, 'B': {'label': ' PAVIMENTO ', 'values': [{'label': 'a', 'value': ' sin reposición ', 'value_norm': 'sin reposición'}], 'label_norm': 'pavimento'}, 'C': {'label': ' CONDICIONES DE EJECUCIÓN', 'values': [{'label': 'c', 'value': 'Cualquier condición de ejecución', 'value_norm': 'cualquier condición de ejecución'}], 'label_norm': 'condiciones de ejecución'}}","[sin reposición, cualquier condición de ejecución


Preview generated. To save outputs, set SAVE=True and re-run this cell.
Saved Parquet to: /work/data/processed
